# Texto en Tablas

En este notebook exploraremos algunas visualizaciones donde el texto es protagonista a través del dataset [guaguas](https://github.com/rivaquiroga/guaguas) preparado por [Riva Quiroga](https://twitter.com/rivaquiroga). Algunos de los análisis están inspirados en los ejemplos que ella incluyó en el repositorio de guaguas.

Para ejecutar el último análisis necesitaremos otras bibliotecas que no estaban en el entorno. Se pueden instalar así:

```
mamba install transformers einops alphashape

pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
```

In [ ]:
from pathlib import Path
from chiricoca.config import setup_style

setup_style(dpi=192)

GUAGUAS_PATH = Path("data") / "guaguas"
GUAGUAS_PATH

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from chiricoca.base.weights import normalize_rows

import matplotlib as mpl
import seaborn as sns

In [ ]:
guaguas = pd.read_csv(GUAGUAS_PATH / '1920-2021.csv.gz')
guaguas.head()

In [ ]:
total_n = guaguas.groupby("nombre")["n"].sum().sort_values(ascending=False)
total_n.head(15)

In [ ]:
total_n[total_n == 1].sample(10)

In [ ]:
total_n.sum()

In [ ]:
total_n[total_n > 10000]

In [ ]:
total_n.plot(kind='hist', bins=100, logy=True, linewidth=0.5, edgecolor='black')

In [ ]:
cumsum_names = total_n.cumsum() / total_n.sum()
cumsum_names

In [ ]:
ax = cumsum_names.reset_index(drop=True).plot(logx=True)
ax.axhline(y=0.5, linestyle='dotted', color='grey')
ax.axhline(y=0.9, linestyle='dotted', color='grey')

In [ ]:
total_n[cumsum_names < 0.90].index

## ¿Hay tendencias temporales en los nombres?

In [ ]:
tabla_anual = (
    guaguas[guaguas["nombre"].isin(total_n[cumsum_names < 0.90].index)]
    .groupby(["anio", "nombre"])["n"]
    .sum()
    .unstack(fill_value=0)
)

tabla_anual


In [ ]:
sns.heatmap(tabla_anual.T.pipe(np.sqrt))

In [ ]:
sns.heatmap(tabla_anual.pipe(np.sqrt).T.sort_values(2021))

In [ ]:
from chiricoca.base.weights import tfidf

In [ ]:
sns.heatmap(tabla_anual.pipe(tfidf).T.sort_values(2021))

In [ ]:
fig, axes = plt.subplots(2, 2)

tabla_anual[['María', 'Emma', 'Rayen']].plot(ax=axes[0][0])
tabla_anual.pipe(tfidf)[['María', 'Emma', 'Rayen']].plot(ax=axes[0][1])

tabla_anual.pipe(lambda x: np.log(x+1)).pipe(normalize_rows)[['María', 'Emma', 'Rayen']].plot(ax=axes[1][0])
tabla_anual.pipe(lambda x: tfidf(x, norm='l1', smooth_idf=True, sublinear_tf=True))[['María', 'Emma', 'Rayen']].plot(ax=axes[1][1])

In [ ]:
our_tfidf = lambda x: tfidf(x, norm='l1', smooth_idf=True, sublinear_tf=True)

fig, ax = plt.subplots(figsize=(12, 12))
sns.heatmap(tabla_anual.pipe(our_tfidf).T.sort_values(2021).tail(100), center=0, yticklabels=True, ax=ax)

In [ ]:
tabla_decadas = (
    tabla_anual.stack()
    .rename("frecuencia")
    .reset_index()
    .assign(decada=lambda x: x["anio"] - (x["anio"] % 10))
    .groupby(["nombre", "decada"])["frecuencia"]
    .sum()
    .unstack()
    .pipe(our_tfidf)
)

sns.heatmap(tabla_decadas, center=0)

In [ ]:
from chiricoca.tables.areas import streamgraph
from matplotlib.colors import rgb2hex

n_name_bins = 10

name_bin = pd.cut(np.log(total_n), n_name_bins, labels=False)

m_colors = list(map(rgb2hex, sns.color_palette("Greens", n_colors=n_name_bins)))
f_colors = list(map(rgb2hex, sns.color_palette("Purples", n_colors=n_name_bins)))

name_to_color = (
    guaguas.groupby(["nombre", "sexo"])["n"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .drop_duplicates(subset=["nombre"])
    .set_index("nombre")["sexo"]
    .to_dict()
)

for k, v in name_to_color.items():
    if v == "M":
        name_to_color[k] = m_colors[name_bin[k]]
    elif v == "F":
        name_to_color[k] = f_colors[name_bin[k]]
    else:
        # caso especial!
        print(k, v)
        name_to_color[k] = '#efefef'


sns.palplot(m_colors)
sns.palplot(f_colors)
# name_to_color

In [ ]:
fig, ax = plt.subplots(figsize=(18, 9.5))

fig.set_facecolor("#efefef")
ax.set_facecolor("#efefef")
ax.set_xlim([1920, 2020])
#ax.set_ylim([0, 1])
ax.set_title("Evolución de Nombres en Chile (1920-2020)", loc="left")
ax.set_ylabel("Proporción de las inscripciones")
ax.set_xlabel("")

streamgraph(
    tabla_anual.fillna(0),
    fig=fig,
    area_colors=name_to_color,
    baseline='zero',
    labels=True,
    label_threshold=1000,
    avoid_label_collisions=True,
    area_args=dict(linewidth=0.01, alpha=0.75),
    ax=ax
)


sns.despine(ax=ax, bottom=True, top=True)

## ¿Existen tendencias asociadas a fenómenos históricos o _pop_?

In [ ]:
tabla_completa = (
    guaguas
    .groupby(["anio", "nombre"])["n"]
    .sum()
    .unstack(fill_value=0)
)

tabla_completa

In [ ]:
def plot_nameseries(names):

    fig, ax = plt.subplots(figsize=(12, 4))

    names.plot(
        ax=ax,
        color=sns.color_palette("plasma", n_colors=len(names.columns)),
        linewidth=2,
    )

    fig.set_facecolor("#efefef")
    ax.set_facecolor("#efefef")
    sns.despine(ax=ax)

    ax.set_xlabel("")
    ax.set_ylabel("# de Registros")

    fig.tight_layout()

    return fig, ax


fig, ax = plot_nameseries(tabla_completa[["Salvador", "Augusto"]])

ax.axvline(1973, linestyle="dotted", linewidth=1, color="black")
ax.annotate(
    "Golpe de Estado\ndirigido por\nAugusto Pinochet,\ninicio de la dictadura",
    xy=(1973.5, 0.99),
    xycoords=("data", "axes fraction"),
    ha="left",
    va="top",
)

ax.axvline(1990, linestyle="dotted", linewidth=1, color="black")
ax.annotate(
    "Regreso a la Democracia",
    xy=(1990.5, 0.99),
    xycoords=("data", "axes fraction"),
    ha="left",
    va="top",
)

ax.axvline(2006, linestyle="dotted", linewidth=1, color="black")
ax.annotate(
    "Muerte\nde Augusto Pinochet",
    xy=(2005.5, 0.8),
    xycoords=("data", "axes fraction"),
    ha="right",
    va="top",
)

ax.scatter(
    [1952, 1958, 1964, 1970],
    tabla_anual.loc[[1952, 1958, 1964, 1970], "Salvador"],
    color="white",
    edgecolor="black",
    label="Elecciones Presidenciales donde participó Salvador Allende",
    zorder=5,
)

ax.legend()
ax.set_title("Uso de los nombres Salvador y Augusto", loc="left")


In [ ]:
fig, ax = plot_nameseries(tabla_completa[["Milenka", "Branco", "Salomé"]].fillna(0))

ax.set_title("Los nombres de Romané (TVN, 2000)", loc="left")


In [ ]:
fig, ax = plot_nameseries(tabla_completa[["Byron"]].fillna(0))
ax.set_title("Everybody (Backstreet's Back)", loc="left")


## ¿Hay tendencias en el tiempo que abarcan los nombres?

$$H = - \sum p_i \log p_i$$

In [ ]:
from scipy.stats import entropy

entropia = (
    tabla_anual
    .fillna(0)
    .apply(entropy)
    .sort_values(ascending=False)
)

entropia


In [ ]:
tabla_anual.idxmax()

In [ ]:
tabla_entropia = (
    tabla_anual.idxmax().rename("anio").to_frame().join(entropia.rename("entropia"))
)

tabla_entropia

In [ ]:
from chiricoca.tables import scatterplot

fig, ax = plt.subplots(figsize=(12, 9))

scatterplot(
    tabla_entropia,
    "anio",
    "entropia",
    annotate=False,
    avoid_collisions=False,
    label_args=dict(fontsize="xx-small"),
    scatter_args=dict(marker=".", color="#abacab"),
    ax=ax
)

tabla_entropia.groupby("anio").mean().rolling(6, center=True).mean().plot(
    ax=ax, color="magenta", linewidth=1
)

scatterplot(
    tabla_entropia.sample(150),
    "anio",
    "entropia",
    annotate=True, avoid_collisions=True,
    label_args=dict(fontsize='x-small', color='black'),
    scatter_args=dict(alpha=0),
    ax=ax
)

## ¿Hay nombres _unisex_? ¿Qué tan _unisex_ son? ¿Cuáles son?

In [ ]:
unisex_names = (
    pd.pivot_table(guaguas, index="nombre", columns="sexo", values="n", aggfunc="sum")
    .join(total_n)
    .fillna(0)
    .assign(mult=lambda x: x["F"] * x["M"])
    .pipe(lambda x: x[(x["mult"] > 0) & (x["n"] > 100)])
    .drop(["mult", "n"], axis=1)
    .pipe(normalize_rows)
    .pipe(lambda x: x[x["F"].between(0.015, 0.985)])
    .join(total_n)
)

unisex_names#.sort_values('n')


In [ ]:
unisex_names["tendency"] = unisex_names["F"] - unisex_names["M"]
unisex_names["tendency"].plot(kind="hist", bins=100)


In [ ]:
from chiricoca.tables.bubbles import bubble_plot

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

bubble_plot(
    unisex_names.reset_index(),
    "tendency",
    "n",
    label_column="nombre",
    palette="cool",
    max_label_size=56,
    starting_y_range=60,
    margin=2,
    ax=ax
)

ax.set_axis_off()
ax.set_title(
    "Nombres compartidos por hombres y mujeres (1920-2020, Registro Civil de Chile)"
)
ax.annotate(
    "Más usado por mujeres →",
    (0.95, 0.01),
    xycoords="axes fraction",
    ha="right",
    va="bottom",
    fontsize="medium",
    color="#abacab",
)
ax.annotate(
    "← Más usado por hombres",
    (0.05, 0.01),
    xycoords="axes fraction",
    ha="left",
    va="bottom",
    fontsize="medium",
    color="#abacab",
)
ax.annotate(
    "Fuente: guaguas, por @RivaQuiroga.",
    (0.5, 0.01),
    xycoords="axes fraction",
    ha="center",
    va="bottom",
    fontsize="medium",
    color="#abacab"
)

fig.set_facecolor("#efefef")
fig.tight_layout()


# ¿Hay _clusters_ de nombres en base a su significado y frecuencia temporal?

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("jinaai/jina-embeddings-v3", trust_remote_code=True)

In [ ]:
model.encode('Pajarito')

In [ ]:
import pandas as pd
df = pd.DataFrame(model.encode(['hombre', 'mujer', 'rey', 'reina']), index=['hombre', 'mujer', 'rey', 'reina'])

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
df_2d = pd.DataFrame(pca.fit_transform(df.values), index=df.index)
df_2d

In [ ]:
scatterplot(df_2d, 0, 1, annotate=True)

In [ ]:
import joblib

if Path('embeddings.pkl.gz').exists():
    embeddings_nombres = joblib.load('embeddings.pkl.gz')
else:
    embeddings_nombres = model.encode(tabla_anual.columns, task='separation', truncate_dim=128)
    joblib.dump(embeddings_nombres, 'embeddings.pkl.gz')

In [ ]:
embeddings_nombres.shape, len(tabla_anual.columns)

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=333*2)
df_nombres_2d = pd.DataFrame(tsne.fit_transform(embeddings_nombres), index=tabla_anual.columns)
df_nombres_2d

In [ ]:
fig, ax = plt.subplots(figsize=(16, 16))

scatterplot(df_nombres_2d, 0, 1, scatter_args=dict(marker='.'), annotate=True, avoid_collisions=False, ax=ax)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_emb = StandardScaler()
scaler_ser = StandardScaler()

embeddings_norm = scaler_emb.fit_transform(embeddings_nombres)
series_norm = scaler_ser.fit_transform(tabla_anual.T)

combined = np.concatenate([embeddings_norm, series_norm], axis=1)
combined.shape

In [ ]:
reducer = TSNE(
    n_components=2
)

embedding_2d = reducer.fit_transform(combined)

In [ ]:
df_combined = pd.DataFrame(embedding_2d, index=tabla_anual.columns)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 16))

scatterplot(df_combined, 0, 1, scatter_args=dict(marker='.'), annotate=True, avoid_collisions=False, ax=ax)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=3,
)

cluster_labels = clusterer.fit_predict(embedding_2d)
df_combined['cluster'] = cluster_labels
df_combined['cluster'].value_counts()

In [ ]:
import alphashape
import numpy as np

fig, ax = plt.subplots(figsize=(16, 12))

# Colores más intuitivos
unique_labels = np.unique(cluster_labels)
real_clusters = unique_labels[unique_labels != -1]
n_clusters = len(real_clusters)

# Paleta de colores distintivos
cluster_colors = sns.color_palette('plasma', n_colors=max(10, n_clusters))
color_map = {}

for i, label in enumerate(real_clusters):
    color_map[label] = cluster_colors[i]

if -1 in unique_labels:
    color_map[-1] = '#333333'

# Plotear puntos
for label in unique_labels:
    mask = cluster_labels == label
    
    if label == -1:
        label_name = f'Ruido ({np.sum(mask)} puntos)'
        alpha = 0.7
        s = 8
    else:
        label_name = f'Cluster {label} ({np.sum(mask)} puntos)'
        alpha = 0.9
        s = 12
    
    ax.scatter(embedding_2d[mask, 0], embedding_2d[mask, 1], 
              c=[color_map[label]], alpha=alpha, s=s, 
              edgecolors='white', linewidth=0.3, label=label_name)

# Dibujar alpha shapes
for label in real_clusters:
    mask = cluster_labels == label
    points = embedding_2d[mask]
    
    if len(points) >= 3:
        alpha_shape = alphashape.alphashape(points, alpha=0.3)
        
        if alpha_shape and hasattr(alpha_shape, 'boundary'):
            if hasattr(alpha_shape.boundary, 'coords'):
                coords = list(alpha_shape.boundary.coords)
                coords_array = np.array(coords)
                ax.plot(coords_array[:, 0], coords_array[:, 1], 
                        color=color_map[label], alpha=0.7, linewidth=2)

# Calcular frecuencias totales de cada nombre
frecuencias_totales = tabla_anual.sum(axis=0)  # suma por columnas (nombres)

# Mostrar top 5 nombres más frecuentes en el centroide de cada cluster
for label in real_clusters:
    mask = cluster_labels == label
    cluster_points = embedding_2d[mask]
    cluster_names = df_combined.index[mask].tolist()
    
    # Calcular centroide del cluster
    centroide = np.mean(cluster_points, axis=0)
    
    # Obtener frecuencias de los nombres en este cluster
    cluster_frequencies = frecuencias_totales[cluster_names]
    
    # Ordenar por frecuencia y tomar top 5
    top_indices = cluster_frequencies.nlargest(5).index
    top_names = top_indices.tolist()
    top_freqs = cluster_frequencies[top_indices].tolist()
    
    # Crear texto con los top 5
    texto_top = []
    for name, freq in zip(top_names, top_freqs):
        texto_top.append(f"{name} ({freq:.0f})")
    
    # Mostrar en el centroide
    texto_completo = "\n".join(texto_top)
    ax.annotate(texto_completo, 
               (centroide[0], centroide[1]),
               xytext=(0, 0), textcoords='offset points',
               fontsize=6, fontweight='bold',
               ha='center', va='center', color='white',
               bbox=dict(boxstyle='round,pad=0.5', 
                        facecolor=color_map[label], alpha=0.8,
                        edgecolor='white', linewidth=1))

ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Clusters HDBSCAN, con nombres más frecuentes por cluster', loc='left')
#ax.grid(True, alpha=0.3)
ax.set_axis_off()
